<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/cambio/Cambio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Câmbio — dólar americano, média mensal (R$/US$)

Motor de coleta e publicação da série `cambio-media-mensal`.

- **Fonte:** Banco Central do Brasil, Sistema Gerenciador de Séries Temporais (SGS), série
  **3698** — "Taxa de câmbio livre - Dólar americano (venda) - média de período - mensal"
  (PTAX venda).
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.3698/dados?formato=json`
- **Definição:** taxa de câmbio **nominal** do dólar americano, mercado livre/comercial,
  cotação de **venda** (PTAX), em reais por dólar, **média aritmética das cotações diárias do
  mês**. É série oficial já agregada: o notebook não calcula média nenhuma, só busca, recorta
  os meses fechados e formata.
- **Unidade publicada:** `moeda` (`R$/US$`, 4 casas), `eixo: reais`, `medida: nivel`,
  `granularidade: mensal`.
- **Período publicado:** `1995-01-01` até o último mês **fechado**.
- **Arquivo gerado:** `indicadores/cambio/cambio-media-mensal.json`.
- **Atribuição:** Banco Central do Brasil, dados abertos (SGS).

## O que este número é — e o que não é

O valor é **nominal**: reais por dólar, sem deflacionar. A série sobe cerca de 6× entre 1995 e
2026, mas parte desse movimento é a inflação brasileira, não uma perda de valor do real frente
ao dólar em termos reais. Quem mede competitividade é o **câmbio real efetivo**, que o BCB
publica em outra série (um índice) e que este notebook **não** calcula.

Também **não** é:

- **câmbio real efetivo** nem índice de competitividade — é uma razão de preços nominais;
- **dólar à vista do fechamento** — a média representa o mês inteiro; o fechamento é um único
  dia e chega a ficar 32% acima da média em jan/1999 (1,9832 no fim do período contra 1,5019
  na média do mês, SGS 3696 × 3698);
- **dólar paralelo ou turismo** — é a PTAX do mercado livre/comercial.

## Por que a média, e não o fechamento

O fechamento do mês é um único dia: uma intervenção ou um leilão de última hora vira o número
do mês inteiro. A média aritmética das cotações diárias é a série que o BCB publica como
"média de período" e representa o mês. A alternativa (SGS 3696, fim de período) teria outro
`id` e outra leitura — "o dólar fechou o mês em…" — e não é esta série.

## Por que começa em 1995-01

A fonte tem leituras desde fev/1953, mas o valor é nominal na moeda corrente: antes de jul/1994
a série atravessa cruzeiro, cruzado, cruzado novo, cruzeiro e cruzeiro real, com saltos de
unidade no meio (entre jul/1993 e ago/1993 e entre jun/1994 e jul/1994).
Juntar essas moedas no mesmo eixo de `R$/US$` seria afirmar continuidade que não existe. O
recorte começa em `1995-01-01`, a mesma janela das demais famílias (IPCA e IGP-M), já inteira
em real.


## Coleta

Leitura integral a cada execução — nada de atualização incremental e nada de ler o JSON já
publicado para decidir o período. O SGS limita apenas as séries **diárias** a janelas de 10
anos; a 3698 é mensal e vem inteira numa requisição. O serviço falha de forma intermitente com
`502`, `ReadTimeout` e até **200 com HTML** no lugar do JSON, então a coleta repete com espera
crescente; o resultado é determinístico quando a requisição passa.


In [1]:
from datetime import date
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "cambio"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("CAMBIO_SAIDA", PASTA))
ARQUIVO = DESTINO / "cambio-media-mensal.json"

ID = "cambio-media-mensal"
SERIE = 3698
URL = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE}/dados?formato=json"

# Recorte editorial: jan/1995, a mesma janela de IPCA e IGP-M. A fonte começa em fev/1953, mas
# o valor é nominal em moeda corrente: antes de jul/1994 a série atravessa cruzeiro, cruzado,
# cruzado novo, cruzeiro e cruzeiro real, com saltos de unidade em 1993 e 1994.
INICIO = date(1995, 1, 1)
HOJE = date.today()
PRIMEIRO_DIA_MES_CORRENTE = HOJE.replace(day=1)


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


print(f"Pasta de saída: {relativo(DESTINO)}")


Pasta de saída: indicadores/cambio


In [2]:
def baixar(tentativas=4):
    # Busca a série inteira (mensal não tem o limite de janela das séries diárias) e devolve
    # [(date, float)] ascendente, uma leitura por mês. Repete em erro HTTP e em 200 com HTML:
    # nesse caso `resposta.json()` estoura ValueError, que também é capturado aqui.
    bruto = None
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(URL, timeout=60)
            resposta.raise_for_status()
            bruto = resposta.json()
            break
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)

    if not bruto:
        raise ValueError(f"a série SGS {SERIE} não retornou dados")

    leituras = {}
    for item in bruto:
        dia, mes, ano = (int(parte) for parte in item["data"].split("/"))
        leituras[date(ano, mes, dia)] = float(item["valor"])
    return sorted(leituras.items())


bruto_sgs = baixar()

print(f"SGS {SERIE}: {len(bruto_sgs)} leituras mensais "
      f"({bruto_sgs[0][0]:%Y-%m} a {bruto_sgs[-1][0]:%Y-%m})")


SGS 3698: 885 leituras mensais (1953-02 a 2026-10)


## Recorte e meses fechados

Duas operações visíveis: (1) cortar a fonte em `1995-01-01`; (2) descartar o mês corrente, que
ainda está aberto — a média do mês só existe depois do último dia. O mês corrente poderia
aparecer na fonte parcial: o Apurômetro publica outubro até o dia 27 e por isso dá 5,09 no
lugar de 5,1294 (outubro fechado); publicar período aberto é o erro que este corte impede.

As datas da fonte já vêm no primeiro dia do mês; a célula confere isso, a ordem e a ausência de
lacuna antes de publicar.


In [3]:
fonte = [(dia, valor) for dia, valor in bruto_sgs if dia >= INICIO]
fechados = [(dia, valor) for dia, valor in fonte if dia < PRIMEIRO_DIA_MES_CORRENTE]

if not fechados:
    raise ValueError("nenhum mês fechado no período publicado")

# C15: um ponto por mês, no primeiro dia, estritamente ascendente, sem lacuna.
assert all(dia.day == 1 for dia, _ in fechados), "data fora do primeiro dia do mês"
assert all(
    (b.year * 12 + b.month) - (a.year * 12 + a.month) == 1
    for (a, _), (b, _) in zip(fechados, fechados[1:])
), "mês faltando, repetido ou fora de ordem"
assert fechados[0][0] == INICIO, f"primeiro mês inesperado: {fechados[0][0]:%Y-%m}"

print(f"Fonte bruta: {len(bruto_sgs)} meses, {bruto_sgs[0][0]:%Y-%m} a {bruto_sgs[-1][0]:%Y-%m} "
      f"(antes de 1995-01 em moeda anterior ao real)")
print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Publicado: {len(fechados)} meses, {fechados[0][0]:%Y-%m} a {fechados[-1][0]:%Y-%m}")
print(f"Primeiro: {fechados[0][0]:%Y-%m} = R$ {fechados[0][1]:.4f}/US$")
print(f"Último:   {fechados[-1][0]:%Y-%m} = R$ {fechados[-1][1]:.4f}/US$")


Fonte bruta: 885 meses, 1953-02 a 2026-10 (antes de 1995-01 em moeda anterior ao real)
Mês corrente excluído: 2026-11
Publicado: 382 meses, 1995-01 a 2026-10
Primeiro: 1995-01 = R$ 0.8471/US$
Último:   2026-10 = R$ 5.1294/US$


## Escrita do JSON do contrato

O arquivo é gravado inteiro — `versao`, `id`, `unidade`, `eixo`, `medida`, `granularidade`,
`fonte`, `urlFonte` e `dados` — sem ler nem preservar nada do que está publicado. O valor vai
como veio da fonte, sem arredondamento novo: `unidade.casas` (4) é só exibição.


In [4]:
serie = {
    "versao": 1,
    "id": ID,
    "unidade": {"tipo": "moeda", "simbolo": "R$/US$", "casas": 4},
    "eixo": "reais",
    "medida": "nivel",
    "granularidade": "mensal",
    "fonte": "Banco Central do Brasil (SGS 3698 — PTAX venda, média mensal)",
    "urlFonte": URL,
    "dados": [
        {"data": dia.strftime("%Y-%m-%d"), "valor": float(valor)}
        for dia, valor in fechados
    ],
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as destino:
    json.dump(serie, destino, ensure_ascii=False, indent=2)
    destino.write("\n")

print(f"{len(serie['dados'])} pontos -> {relativo(ARQUIVO)}")
print(f"  {serie['dados'][0]}")
print(f"  {serie['dados'][-1]}")


382 pontos -> indicadores/cambio/cambio-media-mensal.json
  {'data': '1995-01-01', 'valor': 0.8471}
  {'data': '2026-10-01', 'valor': 5.1294}


## Validação inline

A célula falha se o contrato, o C15, a ordem, a unicidade e a continuidade dos meses, a
finitude dos valores, o primeiro mês ou a contagem divergirem da fonte filtrada. A conferência
é contra a fonte desta execução, não contra números fixos no código: uma revisão do BCB muda o
valor publicado, mas não pode mudar a contagem, o primeiro mês nem a correspondência 1:1 com o
recorte.


In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
UNIDADE = {"tipo": "moeda", "simbolo": "R$/US$", "casas": 4}
FONTE = "Banco Central do Brasil (SGS 3698 — PTAX venda, média mensal)"

assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == ID == "cambio-media-mensal", serie["id"]
assert serie["unidade"] == UNIDADE, serie["unidade"]
assert serie["eixo"] == "reais", serie["eixo"]
assert serie["medida"] == "nivel", serie["medida"]
assert serie["granularidade"] == "mensal", serie["granularidade"]
assert serie["fonte"] == FONTE, serie["fonte"]
assert serie["urlFonte"] == URL, serie["urlFonte"]

dados = serie["dados"]
assert dados, "série vazia"
for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert isinstance(ponto["data"], str), ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto
    assert ponto["valor"] > 0, ponto  # reais por dólar é preço positivo

dias = [date.fromisoformat(ponto["data"]) for ponto in dados]
assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem ou repetidas"
assert all(dia.day == 1 for dia in dias), "data fora de C15 (mensal = AAAA-MM-01)"
assert all(
    (b.year * 12 + b.month) - (a.year * 12 + a.month) == 1 for a, b in zip(dias, dias[1:])
), "mês faltando ou repetido"
assert dados[0]["data"] == "1995-01-01", dados[0]["data"]
assert dias[-1] < PRIMEIRO_DIA_MES_CORRENTE, dados[-1]["data"]

# A série publicada é exatamente o recorte filtrado da fonte: mesma contagem, mesmas datas,
# mesmos valores, sem sobra nem falta.
assert len(dados) == len(fechados), (len(dados), len(fechados))
assert [(date.fromisoformat(ponto["data"]), ponto["valor"]) for ponto in dados] == fechados, (
    "publicado difere do recorte da fonte"
)
assert dados[-1]["data"] == fechados[-1][0].isoformat()
assert dados[-1]["valor"] == fechados[-1][1]

with ARQUIVO.open(encoding="utf-8") as origem:
    assert json.load(origem) == serie, "arquivo gravado difere do objeto validado"

print(f"ok {serie['id']}: {len(dados)} meses, {dados[0]['data']} a {dados[-1]['data']}, "
      f"último R$ {dados[-1]['valor']:.4f}/US$")
print("\ntudo certo: PTAX venda, média mensal, nominal, meses fechados")


ok cambio-media-mensal: 382 meses, 1995-01-01 a 2026-10-01, último R$ 5.1294/US$

tudo certo: PTAX venda, média mensal, nominal, meses fechados
